Для начала исследования набора данных необходимо импортировать и прочитать датасет. Будем использовать датасет `gemstock_clean`, так как `kolvika_clean` гораздо менее информативна.

Посмотрим на размеры набора данных и типы данных, которые есть в нём на данный момент.

In [24]:
import pandas as pd

df2 = pd.read_csv('C://data_parser/data/clean_data/gemstock_clean.csv')

print(df2.shape)
print(df2.head().dtypes)
print(df2.head())

(5930, 16)
gem_weight          object
gem_completeness    object
commercial_color    object
optical_effect      object
color_fastness      object
gem_sertificate     object
gem_clarity         object
gem_origin          object
gem_enhancement     object
gem_cutting         object
gem_size            object
gem_cut_quality     object
gem_main_color      object
gem_price           object
Ссылка на камень    object
gem_name            object
dtype: object
  gem_weight  gem_completeness             commercial_color  \
0    4.54 ct  одиночный камень          Golden,Vivid Yellow   
1    5.05 ct  одиночный камень  Vivid Red,Pigeons Blood Red   
2   15.04 ct  одиночный камень          Golden,Vivid Yellow   
3    1.21 ct  одиночный камень            Vivid orangy-Pink   
4    5.26 ct  одиночный камень                    Neon Blue   

              optical_effect color_fastness     gem_sertificate  \
0  Флюоресценция (умеренная)  Цвет стабилен  Bellerophon Gemlab   
1    Флюоресценция (сильная)  

Как видно из вывода, мой датасет содержит 5930 записей и 16 столбцов.

Первой проблемой, которая бросается в глаза, является то, что все данные приравнены к `object`.

Второй проблемой, является то, что в серии `gem_size` размеры перечислены. Их лучше разбить.

Третьей проблемой является вид `gem_price`. В данный момент там содержится информация об общей цене на камень и одновременно цене за карат, лучше будет разбить данную серию на отдельные, допустим `gem_price`, `gem_price_per_ct` и привести к числовому типу данных.

Четвёртой проблемой является вид `gem_name` в котором ожидается только название камня, но прописаны ещё и его характеристики, это плохо, нужно будет оставить только название.

Пятой проблемой является вид `gem_enhancement` в котором необходимо привести названия обработки к общему виду на латинице, а также скорее всего разбить на отдельные серии, ведь там могут встречаться сразу несколько обработок на один камень.

Теперь рассмотрим по отдельности каждый столбец, чтобы понять, какие значения и их количество он содержит.

In [22]:
print(df2['gem_enhancement'].value_counts().head(50))

gem_enhancement
отсутствует                                                                          4446
низкотемпературная обработка                                                          912
заполнение маслом (незначительная степень)                                            262
термообработка                                                                        165
заполнение маслом (степень умеренная)                                                  49
заполнение маслом (поверхностное)                                                      11
отсутствует, термообработка                                                             2
заполнение маслом (поверхностное), заполнение маслом (незначительная степень)           2
отсутствует, низкотемпературная обработка                                               1
заполнение полимером (степень значительная)                                             1
заполнение маслом (от незначительной до умеренной степени)                          

Здесь мы видим как раз значение в серии `gem_enhancement` и основную проблему, их может быть несколько, они могут отсутствовать(4446) и их необходимо привести к чему-то общему. \
Думаю, что можно разделить `gem_enhancement` на несколько серий:
- во первых `is_treated`, будет содержать бинарное значение, зависящее от того, облагораживался ли камень;
- во вторых `treatmend_method`, будет содержать наименование метода облагораживания;
- в третьих `treatmend_degree`, будет содержать степень облагораживания, так как в значениях во многих местах указаны степени и они явно будут влиять на цену камня.

In [25]:
print(df2['gem_name'].value_counts().head(50))

gem_name
Лот из граната Родолит в огранке круг 4,26 карата, Танзания                  3
Сет аквамаринов в огранке кушон 1,48 карат, Мадагаскар                       3
Сапфир Royal Blue в огранке овал 0,82 карат, Шри-Ланка                       2
Гранат цаворит Intense Green в огранке кушон 1,69 карата, Танзания           2
Сет аквамаринов в огранке октагон 2,87 карат, Мадагаскар                     2
Радужный лунный камень в огранке сердце 7,04 карат, Мадагаскар               2
Негретый розовый сапфир в огранке овал 2,05 карат, Шри-Ланка                 2
Негретый яркий розово-красный рубин в огранке кушон 2,55 карат, Мозамбик     2
Сапфир Royal Blue в огранке кушон 3,11 карат, Шри-Ланка                      2
Негретый рубин в огранке овал 2,01 карат, Мозамбик                           2
Сапфир Royal Blue в огранке кушон 5,01 карат, Шри-Ланка                      2
Пурпурный гранат в огранке кушон 4,95 карат, Африка                          2
Негретый сапфир Teal в огранке кушон 7,05 к

Из вывода видно, что серию `gem_name` также необходимо обработать, ведь она содержит не только название камня, а ещё и его характеристики, которые уже вынесены в отдельные серии.

Необходимо будет с помощью регулярных выражений обработать данную серию и оставить только названия камней. 
Думаю, проще всего это будет сделать с помощью списка названий камней, допустим `gem_names = ['изумруд', 'рубин', 'турмалин']` и т.д.

In [26]:
print(df2['gem_completeness'].value_counts().head(50))

gem_completeness
одиночный камень                                   4490
парные камни                                        830
комплект из трех или более                          428
калиброванные камни                                  50
парные камни, калиброванные камни                    17
комплект из трех или более, калиброванные камни      13
одиночный камень, калиброванные камни                 8
парные боковые камни                                  1
Name: count, dtype: int64


Здесь всплывает ещё одна проблема.. Как видно из вывода выше, комплектность - `gem_completeness`, также может содержать сразу несколько значений и это нужно будет предусмотреть. 
Также привести всё к латинице.

In [29]:
print(df2['gem_price'].value_counts().head(50))

gem_price
44 100 ₽     17
8 900 ₽      16
17 700 ₽     15
13 300 ₽     15
96 900 ₽     13
52 900 ₽     12
35 300 ₽     12
26 500 ₽     11
66 100 ₽     11
39 700 ₽     11
22 100 ₽     11
132 100 ₽    11
79 300 ₽     10
105 700 ₽    10
37 000 ₽     10
7 100 ₽      10
158 500 ₽     9
101 300 ₽     9
57 300 ₽      9
30 900 ₽      8
4 500 ₽       8
228 900 ₽     8
14 100 ₽      8
5 300 ₽       8
6 200 ₽       8
74 900 ₽      7
88 100 ₽      7
10 600 ₽      7
120 600 ₽     7
87 200 ₽      7
162 900 ₽     7
8 000 ₽       7
334 500 ₽     7
48 500 ₽      7
43 200 ₽      7
24 700 ₽      7
41 400 ₽      7
34 400 ₽      6
290 500 ₽     6
33 500 ₽      6
125 900 ₽     6
23 800 ₽      6
277 300 ₽     6
11 500 ₽      6
140 900 ₽     6
281 700 ₽     6
40 500 ₽      6
30 000 ₽      6
20 300 ₽      6
49 300 ₽      6
Name: count, dtype: int64


В выводе информации по серии `gem_price` видно, что во первых, цена ещё не приведена к числовому типу данных, во вторых, встречаются одинаковые цены.. 

В третьих, есть камни, где отдельно рассчитана цена за карат, а есть те, где только общая цена, следовательно цену за карат будет необходимо дополнительно рассчитывать при разделении серии надвое.

Необходимо будет разделить серию надвое, привести к числовому типу данных обе серии, т.е. общую цену в числовое и цену за карат, рассчитываемую на основе общей и веса камня, также в числовое. 



In [30]:
print(df2['gem_clarity'].value_counts().head(50))

gem_clarity
визуально чистый                                                                      3955
с едва заметными включениями                                                           870
с заметными включениями                                                                595
визуально чистый и с едва заметными включениями (для пар)                              179
с едва заметными включениями и визуально чистый (для пар)                              110
рутиловый шелк, эффект бархатистости                                                    69
полупрозрачный                                                                          36
непрозрачный                                                                            17
с сильно заметными включениями                                                          12
визуально чистый, с заметными включениями                                               11
визуально чистый, с едва заметными включениями                                

По `gem_clarity` можно сказать следующее:

- Во первых, необходимо будет привести всё к общим значениям на латинице;
- Во вторых, видно, что присутствуют сразу несколько значений и это будет необходимо регулировать;
- В третьих, видно, что присутствуют отдельные отметки в скобках - для пар, т.е. эти параметры зависят от комплектности (`gem_completeness`).



In [31]:
print(df2['gem_cutting'].value_counts().head(50))

gem_cutting
кушон                                2019
овал                                  991
изумрудная (октагон)                  990
груша                                 647
радиант                               378
сердце                                230
круг                                  209
ашер                                   85
кабошон                                76
фантазийная                            45
триллион                               44
маркиз                                 30
багет                                  23
изумрудная (октагон),радиант           21
принцесса                              20
кабошон,овал                           14
ашер,изумрудная (октагон)              12
кабошон,сахарная голова                10
гексагон                               10
сахарная голова                         7
трапеция                                6
кабошон,фантазийная                     6
кайт,фантазийная                        5
кайт                  

По `gem_cutting` сразу бросается в глаза то, что огранок может быть несколько, судя по всему это также зависит от комплектности.

- Во первых, необходимо будет привести всё к общим наименованиям на латинице;
- Во вторых, понять, как разделять сразу несколько огранок в одной записи.


In [32]:
print(df2['gem_cut_quality'].value_counts().head(50))

gem_cut_quality
Очень хорошее (точная огранка)                                                  2167
Превосходное (высокоточная огранка)                                             1687
Превосходное (высокоточная огранка от GS)                                       1076
Хорошее ("азиатская" или "нативная" огранка)                                     844
Удовлетворительное                                                                 2
Very good                                                                          1
Очень хорошее (точная огранка), Хорошее ("азиатская" или "нативная" огранка)       1
Превосходное (высокоточная огранка), Очень хорошее (точная огранка)                1
Name: count, dtype: int64


По `gem_cut_quality` бросается в глаза опять же, что качеств огранки может быть несколько, хоть их и мало(единичные случаи).

- Во первых, привести всё к общим наименованиям на латинице;
- Во вторых, разобраться с редкими случаями нескольких огранок.



In [33]:
print(df2['gem_origin'].value_counts().head(50))

gem_origin
Шри-Ланка (Цейлон)                        1181
Мадагаскар                                1122
Танзания                                   921
Африка                                     535
Мозамбик                                   446
Бирма (Мьянма)                             357
Афганистан                                 270
Колумбия                                   259
Бразилия                                   134
Нигерия                                    129
Таджикистан (Памир)                         64
Вьетнам                                     51
Замбия                                      37
Камбоджа                                    33
Россия                                      30
Урал, Россия                                30
Конго                                       24
Пакистан                                    17
Кения                                       16
Эфиопия                                     14
Малави                                      12
Ав

По `gem_origin` в глаза бросается опять же по несколько происхождений в одной записи.. 

- Во первых, необходимо будет привести всё к общим наименованиям на латинице;
- Во вторых, понять, что лучше оставлять, конкретное месторождение или только страну, либо разделить на отдельные серии страну и конкретное месторождение.

In [34]:
print(df2['commercial_color'].value_counts().head(50))

commercial_color
Royal Blue / Vivid Blue           174
Vivid Green                       154
Vivid Pink                        126
Cornflower Blue / Intense Blue    109
Vivid Yellow                       99
Teal                               93
Neon Blue                          85
Pigeons Blood Red                  69
Neon Greenish Blue                 68
Intense Green                      64
Muzo Green                         64
Baby Pink                          60
Bi-color                           55
Velvety purple color               51
Pastel Blue                        49
Pastel Yellow                      49
Neon Green                         47
Intense Yellow                     45
Fanta                              45
Intense Blue                       39
Vibrant pink-red                   38
Teal,Bi-color                      36
Lavender                           35
Lagoon                             34
Golden,Vivid Yellow                32
Mint                             

По `commercial_color` пока наиболее хороший результат..

Необходимо будет разобраться со случаями, когда сразу несколько цветов встречаются в одной записи.

In [35]:
print(df2['gem_main_color'].value_counts().head(50))

gem_main_color
зеленый                         789
голубой                         730
синий                           534
розовый                         529
желтый                          394
зеленый,голубой                 270
пурпурный,розовый               241
красный                         232
фиолетовый                      169
пурпурный                       165
оранжевый,розовый               161
желтый,зеленый                  160
красный,розовый                 157
зеленый,синий                   139
голубой,синий                   126
оранжевый                       124
синий,фиолетовый                 94
белый (бесцветный)               89
фиолетовый,розовый               49
оранжевый,желтый                 48
зеленый,розовый                  46
красный,пурпурный                42
красный,оранжевый                38
серый                            38
фиолетовый,пурпурный             36
фиолетовый,серый                 36
синий,серый                      21
голубой,серый

По `gem_main_color` бросается в глаза наличие сразу нескольких цветов в одной записи.

- Во первых, привести все цвета к общим наименованиям на латинице;
- Во вторых, понять, как корректно разделять сразу несколько цветов в одной записи.

In [36]:
print(df2['gem_sertificate'].value_counts().head(50))

gem_sertificate
GFCO GEM LAB - Швейцария                                        1023
AGA                                                              302
Bellerophon Gemlab                                               244
ICA GemLab                                                       196
GRS                                                              155
GAIA (powered by ICA GemLab)                                     125
GIA                                                               73
GUILD                                                             34
МГЛ                                                               32
AIGS                                                              28
Lotus Gemology                                                    17
ICA GemLab & GS Index                                             15
GIT (Thailand)                                                    12
Bellerophon Gemlab, GFCO GEM LAB - Швейцария                       7
ICA GemLab & GS In

По `gem_sertificate` можно заметить, что опять же, присутствует сразу по несколько сертификатов на одну запись, нужно будет разобраться и привести все записи к общему виду на латинице.


In [37]:
print(df2['gem_size'].value_counts().head(50))

gem_size
5 × 3 mm                                      29
3 mm                                           7
2.81 × 2.88 × 2.78 mm                          3
8.64 × 7.59 × 6.08 mm                          3
4 × 3 mm                                       3
5×3                                            3
4.26 × 4.27 × 4.25 mm                          3
6 × 4 mm                                       3
6.92 × 6.74 × 5.03 mm                          2
11.55 × 9.10 × 5.93 mm                         2
6.64 × 6.11 × 4.22 mm                          2
8.94 × 6.83 × 5.86 mm                          2
7.99 × 6.71 × 4.15 mm                          2
6.88 × 6.18 × 4.36 mm                          2
8.51 × 7.55 × 5.40 mm                          2
10.99 × 10.16 × 6.02 mm                        2
5.01 × 4.10 × 2.86 мм5.11 × 4.12 × 2.92 мм     2
9.62 × 8.56 × 6.19 mm                          2
9.94 × 7.81 × 4.59 mm                          2
7.69 × 7.31 × 4.70 mm                          2
12.40 × 9.1

По `gem_size`, как я и говорил ранее, необходимо будет разделить размеры на 3 отдельных серии.

Также встречаются размеры только с одним значением, с двумя и соответственно с тремя.

Встречаются ещё и по несколько размеров в одной записи, необходимо будет разделять.

Также приписка `mm` может встречаться на русском, необходимо будет привести всё к типу данных `float64` при разбиении на серии.

In [38]:
print(df2['optical_effect'].value_counts().head(50))

optical_effect
Флюоресценция (сильная)                                       312
Без флюоресценции                                             179
Флюоресценция (умеренная)                                     178
Флюоресценция (слабая)                                         85
Адуляресценция                                                 35
Флюоресценция (очень сильная)                                  32
Цветовой сдвиг                                                 29
Опалесценция                                                   24
Александритовый эффект (смена цвета)                           16
Астеризм (эффект звезды)                                        6
"Кошачий глаз"                                                  5
Плеохроизм                                                      3
"Mariposa" - Butterfly-wing effect                              2
Флюоресценция (умеренная), Цветовой сдвиг                       2
Флюоресценция (сильная), Астеризм (эффект звезды)            

По `optical_effect` опять же бросается в глаза по несколько значений в одной записи.

- Во первых, привести всё к единому виду наименований на латинице;
- Во вторых, разделить `optical_effect` на две серии - непосредственно оптический эффект и его силу;
- В третьих, разобраться, как разделять эффекты в редких случаях, когда их несколько в записи.


In [39]:
print(df2['color_fastness'].value_counts().head(50))

color_fastness
Цвет стабилен                            257
Не подвергался тестированию               28
"Xамелеон" (обратимый цветовой сдвиг)      7
Name: count, dtype: int64


По `color_fastness` в целом ничего такого в глаза не бросается, необходимо будет привести все наименования к общему виду на латинице и обработать пустые значения (`Не подвергался тестированию`)

In [40]:
print(df2['Ссылка на камень'].value_counts().head(50))

Ссылка на камень
https://gemstock.ru/product/10761/                                                                             1
https://gemstock.ru/product/turmalin-oranzhevato-rozovyy-v-ogranke-oktagon-3-22-karat-afrika/                  1
https://gemstock.ru/product/unheated-light-blue-radiant-cut-sapphire-2-53-ct-sri-lanka/                        1
https://gemstock.ru/product/Bright-greenish-blue-tourmaline-Paraiba-from-Mozambique-3-17-ct/                   1
https://gemstock.ru/product/polikhromnyy-indigolit-v-ogranke-oktagon-4-29-karata/                              1
https://gemstock.ru/product/negretyy-rozovo-fioletovyy-sapfir-v-ogranke-kushon-0-96-karat-madagaskar/          1
https://gemstock.ru/product/purpurnyy-granat-iz-tanzanii-v-ogranke-kushon-4-90-karat/                          1
https://gemstock.ru/product/Unheated-cornflower-blue-sapphire-cushion-cut-1-74-ct-Sri-Lanka/                   1
https://gemstock.ru/product/sero-fioletovaya-birmanskaya-shpinel-v-ogranke-kush

По `Ссылка на камень` необходимо будет удалить, т.к. она не представляет никакой ценности после извлечения данных со странички.

Общие выводы:

- Наиболее часто встречается проблема наличия сразу нескольких значений в одной записи, необходимо будет разобраться, как её решить для каждой серии корректно;
- Все значения всё ещё не приведены к конкретным типам данных и хранятся как `object`, что также необходимо будет исправить и привести серии к нужным типам данных;
- Почти все значения на русском языке, необходимо привести к более лаконичным и кратким аналогиям на латинице.
- Необходимо будет обработать пропуски и удалить ненужную серию `Ссылка на камень`.